# This file is only for testing purposes

---
**Prototyping functions for imdd_lib.py**

In [2]:
import numpy as np
from scipy.special import erfc
import matplotlib.pyplot as plt

from optic.models.devices import mzm, photodiode
from optic.models.channels import linearFiberChannel
from optic.comm.modulation import modulateGray, demodulateGray, grayMapping
from optic.comm.sources import bitSource
from optic.dsp.core import upsample, pulseShape, pnorm, anorm, signalPower
from optic.utils import parameters, dBm2W
from optic.plot import eyediagram

try:
    from optic.dsp.coreGPU import checkGPU
    if checkGPU():
        from optic.dsp.coreGPU import firFilter
    else:
        from optic.dsp.core import firFilter
except ImportError:
    from optic.dsp.core import firFilter

In [8]:
# Utilility functions
def debug_signal(name, x, n=10):
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Shape : {x.shape}")
    print(f"Type  : {x.dtype}")
    print(f"First {n}: {x[:n]}")
    print(f"Mean  : {np.mean(x):.4f}")
    print(f"Std   : {np.std(x):.4f}")
    print(f"Power : {np.mean(np.abs(x)**2):.4f}")

In [22]:
# ── Simulation resolution ────────────────────────────────────────────────────
SpS = 16            # samples per symbol
nBits = 100_000     # bits per simulation run

# --- Data configurations ---
paramBits        = parameters()
paramBits.nBits  = nBits
paramBits.mode   = 'prbs'
paramBits.order  = 23

# --- Pulse configurations ---
paramPulse           = parameters()
paramPulse.pulseType = 'nrz'
paramPulse.SpS       = SpS

# --- Laser configurations ---
paramMZM     = parameters()
paramMZM.Vpi = 2
paramMZM.Vb  = -paramMZM.Vpi / 2

In [29]:
# =============================================================================
# 2. TRANSMITTER BLOCK
# =============================================================================

def build_transmitter(Pi_dBm, M, SpS, Rs, paramBits, paramPulse, paramMZM, seed=None):
    """
    Build the optical transmitter signal chain.

    Bit source → Gray-coded PAM modulation → upsample → NRZ pulse shaping → MZM.

    Parameters
    ----------
    Pi_dBm    : float      — Laser input power to MZM [dBm]
    M         : int        — Modulation order (2 = OOK, 4 = PAM4)
    SpS       : int        — Samples per symbol
    Rs        : float      — Symbol rate [Hz]                                   # currently unused
    paramBits : parameters — Bit source config (nBits, mode, order)
    paramPulse: parameters — Pulse shaping config (pulseType, SpS)
    paramMZM  : parameters — MZM config (Vpi, Vb)
    seed      : int|None   — RNG seed for the bit source (None = not fixed)

    Returns
    -------
    sigTxo : ndarray — Complex optical signal at MZM output
    bitsTx : ndarray — Transmitted bit sequence (reference for BER)
    symbTx : ndarray — Transmitted symbol sequence (reference for PAM4 decision)
    """
    Pi = dBm2W(Pi_dBm)

    if seed is not None:
        paramBits.seed = seed

    # Generate PRBS bit sequence
    bitsTx = bitSource(paramBits)

    # Gray-coded PAM modulation (M=2 → OOK/2-PAM, M=4 → 4-PAM)
    symbTx = modulateGray(bitsTx, M, 'pam')

    debug_signal("After Gray modulation", symbTx)

    symbTx = pnorm(symbTx)  # power normalization
    debug_signal("After power normalization", symbTx)

    # Upsample and NRZ pulse shaping
    symbolsUp = upsample(symbTx, SpS)
    debug_signal("After upsampling", symbolsUp)

    pulse     = pulseShape(paramPulse)
    sigTx     = firFilter(pulse, symbolsUp)
    debug_signal("After pulse shaping", sigTx)

    sigTx     = anorm(sigTx)
    debug_signal("After anorm()", sigTx)

    # Optical intensity modulation via MZM
    Ai     = np.sqrt(Pi)
    sigTxo = mzm(Ai, sigTx, paramMZM)

    return sigTxo, bitsTx, symbTx

In [31]:
# --- Test for OOK ---
Pi_dBm = -10
M = 2               # OOK
Rs = 10e9           # symbol rate [Hz]  / unused
seed = 145          # fixed seed

sigTxo1, bitsTx1, symbTx1 = build_transmitter(
    Pi_dBm, M, SpS, Rs, paramBits, paramPulse, paramMZM, seed=seed
)

#print(f"\nDebug: Numbers of levels and numbers of symbol in each level")
#print(np.unique(symbTx, return_counts=True))


After Gray modulation
---------------------
Shape : (100000,)
Type  : float64
First 10: [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
Mean  : -0.0027
Std   : 1.0000
Power : 1.0000

After power normalization
-------------------------
Shape : (100000,)
Type  : float64
First 10: [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
Mean  : -0.0027
Std   : 1.0000
Power : 1.0000

After upsampling
----------------
Shape : (1600000,)
Type  : float64
First 10: [-1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
Mean  : -0.0002
Std   : 0.2500
Power : 0.0625

After pulse shaping
-------------------
Shape : (1600000,)
Type  : float64
First 10: [-0.0625 -0.0625 -0.0625 -0.0625 -0.0625 -0.0625 -0.0625 -0.0625 -0.0625
 -0.0625]
Mean  : -0.0002
Std   : 0.0565
Power : 0.0032

After anorm()
-------------
Shape : (1600000,)
Type  : float64
First 10: [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
Mean  : -0.0027
Std   : 0.9034
Power : 0.8162


In [33]:
# --- Test for 4-PAM ---
Pi_dBm = -10
M = 4               # 4-PAM
Rs = 10e9           # symbol rate [Hz] / unused
seed = 145          # fixed seed

sigTxo2, bitsTx2, symbTx2 = build_transmitter(
    Pi_dBm, M, SpS, Rs, paramBits, paramPulse, paramMZM, seed=seed
)

#print(f"\nDebug: Numbers of levels and numbers of symbol in each level")
#print(np.unique(symbTx, return_counts=True))


After Gray modulation
---------------------
Shape : (50000,)
Type  : float64
First 10: [-3. -3. -3. -3. -3. -3. -3. -1. -3.  3.]
Mean  : -0.0164
Std   : 2.2359
Power : 4.9995

After power normalization
-------------------------
Shape : (50000,)
Type  : float64
First 10: [-1.34170519 -1.34170519 -1.34170519 -1.34170519 -1.34170519 -1.34170519
 -1.34170519 -0.44723506 -1.34170519  1.34170519]
Mean  : -0.0074
Std   : 1.0000
Power : 1.0000

After upsampling
----------------
Shape : (800000,)
Type  : float64
First 10: [-1.34170519  0.          0.          0.          0.          0.
  0.          0.          0.          0.        ]
Mean  : -0.0005
Std   : 0.2500
Power : 0.0625

After pulse shaping
-------------------
Shape : (800000,)
Type  : float64
First 10: [-0.08385657 -0.08385657 -0.08385657 -0.08385657 -0.08385657 -0.08385657
 -0.08385657 -0.08385657 -0.08385657 -0.08385657]
Mean  : -0.0005
Std   : 0.0564
Power : 0.0032

After anorm()
-------------
Shape : (800000,)
Type  : float64
Fi

In [34]:
# =============================================================================
# 3. FIBER CHANNEL BLOCK
# =============================================================================

def build_channel(sigTxo, paramCh):
    """
    Propagate the optical signal through a linear fiber channel.

    No optical amplification is applied — fiber loss directly reduces received power,
    which is controlled via Pi_dBm at the transmitter to sweep received power levels.

    Parameters
    ----------
    sigTxo  : ndarray    — Complex optical signal at the MZM output
    paramCh : parameters — Fiber channel config:
                               L   [km]       total link length
                               α   [dB/km]    attenuation coefficient
                               D   [ps/nm/km] dispersion parameter
                               Fc  [Hz]       central optical frequency
                               Fs  [Hz]       simulation sampling frequency

    Returns
    -------
    sigCh : ndarray — Complex optical signal after fiber propagation
    """
    sigCh = linearFiberChannel(sigTxo, paramCh)
    return sigCh

In [35]:
# ── Fiber parameters (defaults — overridden per sweep) ───────────────────────
FIBER_L     = 10   # fiber length [km]
FIBER_ALPHA = 0.2  # attenuation [dB/km]
FIBER_D     = 16   # dispersion  [ps/nm/km]
Fc          = 193.1e12  # central optical frequency [Hz]

Fs = Rs * SpS

In [36]:
paramCh    = parameters()
paramCh.L  = FIBER_L
paramCh.α  = FIBER_ALPHA
paramCh.D  = FIBER_D
paramCh.Fc = Fc
paramCh.Fs = Fs

In [39]:
sigCh1 = build_channel(sigTxo1, paramCh)
print("First 16 amplitudes:")
print(np.abs(sigCh1[:16]))

print("First 16 phases:")
print(np.angle(sigCh1[:16]))

First 16 amplitudes:
[0.00246987 0.00202628 0.0015272  0.00133422 0.00109439 0.00088066
 0.00087005 0.00076699 0.00060253 0.00057801 0.00063706 0.00064642
 0.00058868 0.00049439 0.00039379 0.00030436]
First 16 phases:
[ 0.11727724  0.52140871  1.03686192  1.66855117  2.58438003 -2.7822257
 -1.61910466 -0.13424065  1.38096056  2.94389109 -1.41462785  0.82627427
 -3.01652763 -0.43472938  2.25931125 -1.23582109]


In [40]:
sigCh2 = build_channel(sigTxo2, paramCh)
print("First 16 amplitudes:")
print(np.abs(sigCh2[:16]))

print("First 16 phases:")
print(np.angle(sigCh2[:16]))

First 16 amplitudes:
[0.00246988 0.00202627 0.00152721 0.00133421 0.00109439 0.00088067
 0.00087005 0.00076699 0.00060253 0.00057801 0.00063705 0.00064641
 0.00058868 0.00049439 0.00039379 0.00030436]
First 16 phases:
[ 0.11727869  0.52140789  1.0368613   1.66855387  2.5843757  -2.7822232
 -1.61910132 -0.13424559  1.38095675  2.94389773 -1.41462251  0.82627415
 -3.01653249 -0.43473777  2.25930041 -1.2358335 ]


In [41]:
# =============================================================================
# 4. RECEIVER BLOCK
# =============================================================================

def build_receiver(sigCh, bitsTx, M, SpS, paramPD, discard=100):
    """
    Receiver chain: photodiode → normalization → sampling → decision → BER.
    Supports OOK (M=2) and 4-PAM (M=4) via a unified multi-level slicer.

    Parameters
    ----------
    sigCh   : ndarray    — Optical signal at the photodiode input
    bitsTx  : ndarray    — Reference transmitted bit sequence
    M       : int        — Modulation order (2 or 4)
    SpS     : int        — Samples per symbol
    paramPD : parameters — Photodiode config (ideal, B, Fs, ipd_sat, ...)
    discard : int        — Number of symbols to discard at each end when counting BER

    Returns
    -------
    dict with keys:
        'BER'    : float   — Measured bit error rate
        'Pb'     : float   — Approx. theoretical BER from worst-case Q-factor
                             (exact for OOK, approximate for 4-PAM)
        'Q'      : float   — Worst-case (minimum) eye Q-factor across all
                             adjacent decision levels
        'I_Rx'   : ndarray — Full-rate photodiode current (used for eye diagrams)
        'I_dec'  : ndarray — Symbol-rate samples at decision point
        'bitsRx' : ndarray — Decided bit sequence
    """
    if M not in (2, 4):
        raise ValueError(f"Unsupported modulation order M={M}. Use M=2 (OOK) or M=4 (PAM4).")

    # Optical-to-electrical conversion
    I_Rx      = photodiode(sigCh, paramPD)
    I_Rx_full = I_Rx.copy()  # keep full-rate copy for eye diagram

    # Normalize
    I_Rx = I_Rx / np.std(I_Rx)

    # Sample at symbol center (one sample per symbol)
    I_dec = I_Rx[0::SpS]
    print(I_dec[:50])

    symbTx_ideal = pnorm(modulateGray(bitsTx, M, 'pam')).real

    # Normalized levels: for thresholding with I_dec
    levels_norm = np.sort(np.unique(np.round(pnorm(grayMapping(M, 'pam')).real, 10)))

    # Un-normalized levels: for demodulateGray
    levels_raw = np.sort(np.unique(np.round(grayMapping(M, 'pam').real, 10)))

    symb_idx = np.argmin(np.abs(symbTx_ideal[:, None] - levels_norm[None, :]), axis=1)

    means = np.array([I_dec[symb_idx == k].mean() for k in range(M)])
    stds = np.array([I_dec[symb_idx == k].std() for k in range(M)])

    thr = (stds[:-1] * means[1:] + stds[1:] * means[:-1]) / (stds[:-1] + stds[1:])

    decided_idx = np.digitize(I_dec, thr)
    symbDec = levels_raw[decided_idx]  # <-- use RAW levels here
    bitsRx = demodulateGray(symbDec, M, 'pam').astype(int)

    Q = np.min((means[1:] - means[:-1]) / (stds[1:] + stds[:-1]))
    Pb = (2 * (M - 1) / M) * 0.5 * erfc(Q / np.sqrt(2)) / np.log2(M)

    err = np.logical_xor(
        bitsRx[discard: bitsRx.size - discard],
        bitsTx[discard: bitsTx.size - discard],
    )
    BER = np.mean(err)

    return {
        'BER': BER, 'Pb': Pb, 'Q': Q,
        'I_Rx': I_Rx_full, 'I_dec': I_dec, 'bitsRx': bitsRx,
    }

In [42]:
paramPD       = parameters()
paramPD.ideal = False
paramPD.Fs    = Fs
paramPD.B     = Rs

In [44]:
rx_result = build_receiver(sigCh1, bitsTx1, 2, SpS, paramPD, discard=100)

[ 8.10440603e-02 -7.64251717e-02 -7.96477259e-02  8.77728859e-04
 -4.24269449e-02  6.65312833e-02  4.23993195e-02 -2.28794465e-02
  3.11936945e-02  5.14458185e-03 -2.65904278e-02  6.09490627e-02
  8.58514534e-03  2.72945633e-03 -1.38927336e-01  2.43928727e+00
 -1.48531616e-01  7.78417696e-03  2.44358119e+00 -7.65022790e-02
 -6.13036448e-02 -3.84512062e-02  2.50089332e+00 -7.66601834e-02
 -5.09723362e-02 -2.63080680e-02 -1.08201879e-02  1.14776571e-02
  4.08172874e-03  3.15929419e-02 -3.49463088e-02 -6.66143946e-02
 -1.47835996e-01  2.37774795e+00 -5.97395605e-02 -1.20860079e-01
  2.30836102e+00 -3.19077212e-01  2.35060189e+00 -2.42179245e-01
  2.23715503e+00  2.20982100e+00 -7.97316353e-02 -5.03897825e-02
 -1.35966139e-01  2.47208497e+00 -1.03554111e-01  4.55378269e-02
 -5.17686898e-03 -3.81854459e-02]


Testing codes

In [51]:
symbTx_ideal = (modulateGray(bitsTx2, 4, 'pam')).real
print(symbTx_ideal[:50])

[-3. -3. -3. -3. -3. -3. -3. -1. -3.  3. -3.  3. -3. -3. -3. -3. -1. -3.
  3.  3.  1. -3. -1. -3. -3. -1. -3.  3. -3.  3. -1. -3.  3. -3.  1. -3.
  3.  3.  1. -1. -1.  3.  3.  3. -3.  1. -3.  3. -3. -3.]


In [61]:
levels_raw  = np.sort(grayMapping(M, 'pam').real)
levels_norm = pnorm(levels_raw)

print(levels_norm[:50])
print(levels_raw[:50])

[-1.34164079 -0.4472136   0.4472136   1.34164079]
[-3. -1.  1.  3.]
